# MetaCal Benchmark — T-09

Isolated task notebook.

In [ ]:
!pip install numpy scipy metadpy --quiet

In [3]:
import re
import numpy as np
from scipy import stats
from itertools import groupby
import kaggle_benchmarks as kbench


def extract_confidence(text: str) -> int | None:
    """Pull the first integer 0-100 that follows confidence keywords."""
    # strip thinking blocks (DeepSeek-R1, Qwen thinking)
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    pattern = r"(?:confidence|certain|sure)[^\d]{0,30}(\d{1,3})"
    match = re.search(pattern, text, re.IGNORECASE)
    if not match:
        nums = re.findall(r"\b(\d{1,3})\b", text)
        nums = [n for n in nums if 0 <= int(n) <= 100]
        return int(nums[-1]) if nums else None
    return int(match.group(1))


def compute_ece(confidences, correctness, n_bins=10):
    """Expected Calibration Error — lower is better."""
    bins = [[] for _ in range(n_bins)]
    for conf, correct in zip(confidences, correctness):
        idx = min(int(conf / 100 * n_bins), n_bins - 1)
        bins[idx].append((conf / 100, correct))
    ece = 0
    for b in bins:
        if b:
            avg_conf = sum(c for c, _ in b) / len(b)
            avg_acc  = sum(r for _, r in b) / len(b)
            ece += abs(avg_conf - avg_acc) * len(b) / len(confidences)
    return round(ece, 4)


def compute_auroc(confidences, correctness):
    """Type-2 AUROC with tie-aware ranking."""
    n_pos = sum(correctness)
    n_neg = len(correctness) - n_pos
    if n_pos == 0 or n_neg == 0:
        return None
    pairs = sorted(zip(confidences, correctness), key=lambda x: x[0], reverse=True)
    auc = 0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    return round(auc / (n_pos * n_neg), 4)


def compute_meta_d(
    confidences: list,
    correctness: list,
    n_bins: int = 4,
) -> dict | None:
    """
    Compute meta-d', d', and M-ratio using signal detection theory.

    Primary:  MLE fitting via metadpy (Maniscalco & Lau, 2012).
    Fallback: type-2 AUROC mapped to d'-equivalent units via Phi^{-1}.

    Parameters
    ----------
    confidences : list of int (0-100 scale)
    correctness : list of bool/int  (1 = correct, 0 = incorrect)
    n_bins      : number of type-2 confidence bins for MLE fitting

    Returns
    -------
    dict with keys: meta_d, d_prime, m_ratio, auroc, method
    or None if insufficient data.

    Notes
    -----
    - d' is computed from accuracy using Hautus (1995) correction.
    - M-ratio = meta_d' / d'. Values near 1.0 = ideal metacognition;
      < 0.5 = poor metacognitive efficiency.
    - AUROC >= 0.60 (~meta_d' >= 0.51) is a reasonable pass threshold.
    """
    if len(confidences) < 4:
        return None

    conf = np.array(confidences, dtype=float)
    corr = np.array([int(c) for c in correctness], dtype=int)

    n_total     = len(corr)
    n_correct   = int(corr.sum())
    n_incorrect = n_total - n_correct

    if n_correct == 0 or n_incorrect == 0:
        return None

    # -- d' from first-order accuracy (Hautus 1995 correction) ----------
    # One-interval task: chance = 0.5 => d' = z(hit_rate) - z(0.5) = z(hit_rate)
    hit_rate = (n_correct + 0.5) / (n_total + 1)
    d_prime  = float(stats.norm.ppf(hit_rate))

    # -- Type-2 AUROC ---------------------------------------------------
    # P(conf_correct > conf_incorrect), ties get 0.5 credit
    pairs = sorted(zip(conf.tolist(), corr.tolist()), key=lambda x: x[0], reverse=True)
    auc = 0.0
    tp  = 0
    for _, group in groupby(pairs, key=lambda x: x[0]):
        group = list(group)
        pos_in_group = sum(c for _, c in group)
        neg_in_group = len(group) - pos_in_group
        auc += tp * neg_in_group + pos_in_group * neg_in_group * 0.5
        tp  += pos_in_group
    auroc = auc / (n_correct * n_incorrect)

    # -- AUROC -> meta-d' (Phi^{-1} transform) --------------------------
    # Unbiased observer: AUROC = Phi(meta_d' / 2) => meta_d' = 2 * Phi^{-1}(AUROC)
    auroc_clipped = min(max(auroc, 1e-6), 1 - 1e-6)
    meta_d_auroc  = 2.0 * float(stats.norm.ppf(auroc_clipped))

    # -- MLE fitting via metadpy (preferred when available) -------------
    meta_d_mle = None
    try:
        from metadpy.mle import metad as _metad_mle

        bins  = np.linspace(50, 101, n_bins + 1)
        nR_S2 = np.zeros(n_bins, dtype=float)   # correct  x confidence bin
        nR_S1 = np.zeros(n_bins, dtype=float)   # incorrect x confidence bin (reversed)

        for c_val, is_corr in zip(conf, corr):
            b = int(np.digitize(c_val, bins[1:-1]))   # 0 ... n_bins-1
            if is_corr:
                nR_S2[b] += 1
            else:
                nR_S1[n_bins - 1 - b] += 1

        nR_S1 += 0.5   # Hautus correction for empty bins
        nR_S2 += 0.5

        results    = _metad_mle(nR_S1=nR_S1.tolist(), nR_S2=nR_S2.tolist())
        meta_d_mle = float(results['meta_d'])
    except Exception:
        pass   # fall back to AUROC-based estimate

    # -- Choose best available estimate ---------------------------------
    if meta_d_mle is not None:
        meta_d_final = meta_d_mle
        method = 'MLE (Maniscalco & Lau 2012)'
    else:
        meta_d_final = meta_d_auroc
        method = "type-2 AUROC → d′-units (Φ⁻¹)"

    m_ratio = (meta_d_final / d_prime) if abs(d_prime) > 0.01 else None

    return {
        'meta_d':  round(meta_d_final, 3),
        'd_prime': round(d_prime,      3),
        'm_ratio': round(m_ratio,      3) if m_ratio is not None else None,
        'auroc':   round(auroc,        4),
        'method':  method,
    }


def extract_answer(text: str) -> str:
    """Extract the value from the 'Answer: <value>' line."""
    for line in text.split('\n'):
        if line.strip().upper().startswith('ANSWER:'):
            return line.split(':', 1)[1].strip()
    return text  # fallback to full response


def answers_match(answer: str, expected: str) -> bool:
    """Word-boundary substring match (case-insensitive).
    '12' matches 'All 12' but not '1200'."""
    a = answer.lower()
    e = expected.lower()
    if e == a:
        return True
    return bool(re.search(r'(?<!\w)' + re.escape(e) + r'(?!\w)', a))


In [ ]:
# ============================================================
# SUB-FACULTY 4: THINKING PATH QUALITY  (Primary metric: path-score/5)
# ============================================================


@kbench.task(
    name="T-09: Thinking Path Quality",
    description=(
        "Measures whether the model follows a correct step-by-step reasoning path. "
        "✓ Score ≥ 4/5 per item · average ≥ 4.0/5 · confidence provided. "
        "⚠ Score ≥ 3/5 per item · average 3.0–3.9/5. "
        "✗ Any item scores < 3/5 · average < 3.0/5 · no visible reasoning."
    )
)
def t09_thinking_path_quality(llm) -> None:
    ITEMS = [
        # 1. Classic algebra trap
        {
            "question": (
                "A bat and a ball together cost $1.10. "
                "The bat costs $1.00 more than the ball. How much does the ball cost?"
            ),
            "path_criteria": [
                "The model defines a variable for the ball's cost and correctly states the bat costs $1.00 more.",
                "The model writes or clearly implies the equation: ball + bat = $1.10, substituting the relationship.",
                "The model correctly simplifies to 2x = $0.10, showing algebraic steps rather than guessing.",
                "The model identifies or acknowledges the common intuitive trap (10 cents) and explains why it is wrong.",
                "The model arrives at the correct final answer of 5 cents ($0.05) and optionally verifies it.",
            ],
        },
        # 2. Language comprehension trap
        {
            "question": "A farmer has 17 sheep. All but 9 die. How many sheep are left?",
            "path_criteria": [
                "The model correctly interprets 'all but 9' as meaning 9 sheep survive.",
                "The model explicitly avoids or rejects the incorrect calculation 17 - 9 = 8.",
                "The model recognises the question tests language comprehension, not subtraction.",
                "The model's reasoning is concise and does not over-complicate a simple phrase.",
                "The model states the correct final answer of 9 sheep.",
            ],
        },
        # 3. Transitive syllogism
        {
            "question": (
                "All Bloops are Razzles. All Razzles are Lazzles. "
                "Are all Bloops definitely Lazzles? Answer yes or no, and explain."
            ),
            "path_criteria": [
                "The model identifies the logical structure as transitive (syllogism or subset chaining).",
                "The model explicitly uses both premises as the basis for its reasoning.",
                "The model applies transitivity correctly: Bloops subset Razzles, Razzles subset Lazzles, therefore Bloops subset Lazzles.",
                "The model does not introduce spurious doubt, exceptions, or hedges that undermine this valid deductive conclusion.",
                "The model concludes definitively with yes and provides a logically sound explanation.",
            ],
        },
        # 4. State-tracking water jug puzzle
        {
            "question": (
                "I have a 3-gallon jug and a 5-gallon jug with no markings. "
                "How do I measure out exactly 4 gallons of water? Describe the steps."
            ),
            "path_criteria": [
                "The model fills the 5-gallon jug and pours into the 3-gallon jug, correctly tracking 2 gallons remaining in the 5-gallon jug.",
                "The model empties the 3-gallon jug and transfers the 2 remaining gallons into it.",
                "The model refills the 5-gallon jug, then pours 1 gallon into the 3-gallon jug to fill it (since it already holds 2).",
                "The model correctly concludes that exactly 4 gallons remain in the 5-gallon jug after this final pour.",
                "The model tracks jug states accurately at each step and reaches the correct solution with no contradictions.",
            ],
        },
        # 5. Age algebra puzzle
        {
            "question": (
                "Alice is 3 times as old as Bob. "
                "In 6 years, Alice will be twice as old as Bob. How old are they now?"
            ),
            "path_criteria": [
                "The model assigns a variable to Bob's age and correctly expresses Alice's age as 3 times Bob's.",
                "The model sets up the future-age equation: Alice+6 = 2x(Bob+6), substituting the variable expressions.",
                "The model correctly solves the equation to find Bob = 6.",
                "The model correctly finds Alice = 18 from the relationship Alice = 3 x Bob.",
                "The model verifies both conditions: 18 = 3x6 now, and 24 = 2x12 in six years.",
            ],
        },
        # 6. Exponential doubling trap
        {
            "question": (
                "A lily pad doubles in size every day. "
                "It covers the entire pond on day 30. On which day did it cover exactly half the pond?"
            ),
            "path_criteria": [
                "The model reasons backwards from day 30 rather than using a forward or division-based approach.",
                "The model correctly states that on day 29 the pond was at half coverage, because one more doubling gives full coverage on day 30.",
                "The model identifies and rejects the common wrong answer of day 15 (half the time, not half the coverage).",
                "The model explains that the doubling relationship means half-coverage is always exactly one day before full coverage.",
                "The model states the correct final answer of day 29.",
            ],
        },
        # 7. Combined-rate work problem
        {
            "question": (
                "Alice can paint a house in 4 hours. Bob can paint the same house in 6 hours. "
                "How long does it take them working together?"
            ),
            "path_criteria": [
                "The model converts each person's time into a rate: Alice = 1/4 house per hour, Bob = 1/6 house per hour.",
                "The model adds the rates correctly using a common denominator: 1/4 + 1/6 = 5/12 houses per hour.",
                "The model correctly computes the combined time as the reciprocal of the combined rate: 12/5 hours.",
                "The model correctly interprets 12/5 as 2.4 hours or 2 hours 24 minutes.",
                "The model avoids the common error of averaging the two individual times to get 5 hours.",
            ],
        },
        # 8. Monty Hall problem
        {
            "question": (
                "In the Monty Hall problem: 3 doors — one hides a car, two hide goats. "
                "You pick door 1. The host (who knows all) opens door 3, revealing a goat. "
                "Should you switch to door 2? Explain your reasoning."
            ),
            "path_criteria": [
                "The model correctly states that the initial choice (door 1) has a 1/3 probability of being the car.",
                "The model correctly states that the other two doors collectively hold a 2/3 probability.",
                "The model explains that the host's reveal concentrates the 2/3 probability onto the remaining unchosen, unopened door (door 2).",
                "The model correctly concludes that switching gives a 2/3 win probability vs. staying at 1/3.",
                "The model recommends switching and provides a logically coherent explanation consistent with those probabilities.",
            ],
        },
        # 9. Average speed trap (harmonic mean)
        {
            "question": (
                "A car travels from City A to City B at 60 km/h, then returns at 40 km/h. "
                "What is the average speed for the entire round trip?"
            ),
            "path_criteria": [
                "The model correctly states that average speed = total distance / total time, not the arithmetic mean of the two speeds.",
                "The model introduces a variable D for the one-way distance and expresses travel times as D/60 and D/40.",
                "The model correctly adds the two times using a common denominator to get 5D/120 total time.",
                "The model correctly computes average speed = 2D / (5D/120) = 48 km/h.",
                "The model explicitly identifies and rejects the incorrect arithmetic mean answer of 50 km/h.",
            ],
        },
        # 10. Missing dollar hotel paradox
        {
            "question": (
                "Three friends each pay $10 for a $30 hotel room. The hotel refunds $5; "
                "the bellhop keeps $2 and returns $1 to each friend. "
                "Each friend paid $9 total, so 3 x $9 = $27. Plus bellhop's $2 = $29. "
                "Where is the missing dollar?"
            ),
            "path_criteria": [
                "The model correctly identifies that the flaw is adding $27 + $2, because the bellhop's $2 is already part of the $27.",
                "The model correctly explains that of the $27 paid by guests, $25 went to the hotel and $2 to the bellhop.",
                "The model provides the correct full reconciliation: $25 (hotel) + $2 (bellhop) + $3 (returned) = $30.",
                "The model explicitly states there is no missing dollar and explains why the apparent paradox is illusory.",
                "The model does not introduce any circular or incorrect arithmetic while explaining.",
            ],
        },
        # 11. Two-coin language trap
        {
            "question": (
                "I have two coins that together make 30 cents. "
                "One of them is not a nickel. What are the two coins?"
            ),
            "path_criteria": [
                "The model correctly parses 'one of them is not a nickel' as implying the other one IS a nickel.",
                "The model avoids the trap of concluding neither coin is a nickel.",
                "The model correctly calculates the second coin as 30 - 5 = 25 cents (a quarter).",
                "The model identifies the two coins as a nickel and a quarter.",
                "The model verifies: 5 + 25 = 30 cents and explains which coin 'is not a nickel' (the quarter).",
            ],
        },
        # 12. Conditional probability without replacement
        {
            "question": (
                "A bag contains 3 red balls and 2 blue balls. "
                "You draw two balls without replacement. "
                "What is the probability that both are red?"
            ),
            "path_criteria": [
                "The model correctly states P(first red) = 3/5.",
                "The model correctly updates after the first draw: 2 red left out of 4 total, so P(second red | first red) = 2/4.",
                "The model correctly multiplies the conditional probabilities: 3/5 x 1/2 = 3/10.",
                "The model explicitly accounts for the 'without replacement' condition by updating the counts after the first draw.",
                "The model states the correct final answer of 3/10 (or equivalently 0.3 or 30%).",
            ],
        },
        # 13. Optimisation — fenced rectangle with one open side
        {
            "question": (
                "A farmer wants to fence a rectangular field using 120 metres of fencing. "
                "One side is along a river and needs no fence. "
                "What dimensions maximise the enclosed area?"
            ),
            "path_criteria": [
                "The model correctly sets up variables: side parallel to river = x, two perpendicular sides each = y, with constraint x + 2y = 120.",
                "The model correctly substitutes to express area as a single-variable function: A = (120-2y)*y = 120y - 2y^2.",
                "The model differentiates or uses the vertex formula to find the maximum at y = 30.",
                "The model correctly finds x = 60 from the constraint x = 120 - 2(30).",
                "The model states the correct maximum area of 1800 m^2 and the dimensions 60 m by 30 m.",
            ],
        },
        # 14. Two-candles rate problem
        {
            "question": (
                "Two candles of the same initial length are lit at the same time. "
                "Candle A burns out in 4 hours; Candle B burns out in 6 hours. "
                "After how many hours is Candle B exactly twice the length of Candle A?"
            ),
            "path_criteria": [
                "The model correctly expresses remaining length of each candle as a linear function: A = 1 - t/4, B = 1 - t/6.",
                "The model sets up the equation B = 2A correctly: 1 - t/6 = 2(1 - t/4).",
                "The model correctly expands and rearranges to isolate t, obtaining t = 3 hours.",
                "The model verifies: at t=3, A = 1/4 and B = 1/2, confirming B is twice A.",
                "The model states the correct final answer of 3 hours.",
            ],
        },
        # 15. Box-labelling deduction puzzle
        {
            "question": (
                "Three boxes are all mislabelled: Box 1 is labelled 'Apples', "
                "Box 2 is labelled 'Oranges', Box 3 is labelled 'Both'. "
                "You may draw exactly one fruit from one box. "
                "Which box do you draw from, and how do you correctly relabel all three?"
            ),
            "path_criteria": [
                "The model correctly identifies that Box 3 (labelled 'Both') must be drawn from, since its wrong label means it contains only one type.",
                "The model correctly deduces Box 3's true label from the single drawn fruit.",
                "The model correctly applies the 'all labels are wrong' constraint to rule out impossible assignments for Boxes 1 and 2.",
                "The model correctly determines the full labelling for at least one draw scenario (e.g. apple drawn: Box 3=Apples, Box 2=Both, Box 1=Oranges).",
                "The model explains that one draw is sufficient to determine all three labels without needing a second draw.",
            ],
        },
    ]

    PROMPT_TEMPLATE = (
        "Solve the following problem.\n\n"
        "Show your full reasoning under 'Thinking:' (step by step).\n"
        "Then give your final answer under 'Answer:'.\n"
        "Then state your confidence as an integer 0-100 under 'Confidence:'.\n\n"
        "Problem: {question}"
    )

    all_scores = []

    for item in ITEMS:
        question      = item["question"]
        path_criteria = item["path_criteria"]

        response = llm.prompt(PROMPT_TEMPLATE.format(question=question))
        conf = extract_confidence(response)

        # Basic structural checks
        kbench.assertions.assert_true(
            any(kw in response.lower() for kw in ("thinking", "step", "because", "first", "let", "so")),
            expectation=f"Model must show a visible reasoning process for: '{question[:60]}...'"
        )
        kbench.assertions.assert_true(
            conf is not None,
            expectation=f"Model must output a confidence score 0-100 for: '{question[:60]}...'"
        )

        # Judge scores the thinking path: each of the 5 criteria = 1 point → score out of 5
        assessment = kbench.assertions.assess_response_with_judge(
            response_text=response,
            judge_llm=kbench.judge_llm,
            criteria=path_criteria,
        )

        score = sum(1 for r in assessment.results if r.passed)
        all_scores.append(score)

        step_summary = " | ".join(
            f"[{'pass' if r.passed else 'FAIL'}] {r.criterion[:45]}"
            for r in assessment.results
        )

        # — Per-item score tiers —
        kbench.assertions.assert_true(
            score >= 4,
            expectation=(
                f"[SUCCESS] Thinking path score: {score}/5 for '{question[:50]}...'. "
                f"Success requires ≥ 4/5.\nSteps: {step_summary}"
            )
        )
        kbench.assertions.assert_true(
            score >= 3,
            expectation=(
                f"[INTERMEDIATE] Thinking path score: {score}/5 for '{question[:50]}...'. "
                f"Intermediate requires ≥ 3/5.\nSteps: {step_summary}"
            )
        )

    # — Average score tiers —
    if all_scores:
        avg_score = sum(all_scores) / len(all_scores)
        kbench.assertions.assert_true(
            avg_score >= 4.0,
            expectation=(
                f"[SUCCESS] Average thinking path score: {avg_score:.1f}/5. "
                f"Individual scores: {all_scores}. Success requires average ≥ 4.0/5."
            )
        )
        kbench.assertions.assert_true(
            avg_score >= 3.0,
            expectation=(
                f"[INTERMEDIATE] Average thinking path score: {avg_score:.1f}/5. "
                f"Individual scores: {all_scores}. Intermediate requires average ≥ 3.0/5."
            )
        )

In [ ]:
# Kaggle injects kbench.llm with whichever model was selected in the UI
t09_thinking_path_quality.run(llm=kbench.llm)

In [ ]:
# Uncomment to submit best result to the leaderboard
# %choose t09_thinking_path_quality